# Train and evaluate WITH SAM
Two-stage lens alignment and VQA training. Training uses original + two SAM views; answer evaluation uses original images only.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the loc_lens repository."
sys.path.insert(0, str(ROOT / "src"))

MANIFEST = ROOT / "data/vqarad/manifest.jsonl"
LENSES = ROOT / "data/vqarad/lenses"

In [2]:
import torch
from argparse import Namespace
from transformers import AutoProcessor, set_seed
from localization_lens.training import MODEL_ID, build_model, run_stage

assert torch.cuda.is_available(), "Select a GPU kernel/environment before training."

/home/hasan/extras/loc_lens/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Editable experiment configuration
Use `SMOKE_TEST=True` for two optimizer steps per stage. Set it false for the full experiment.

In [3]:
SMOKE_TEST = True
OUTPUT = ROOT / "outputs" / ("smolvlm-with-sam-smoke" if SMOKE_TEST else "smolvlm-with-sam")
args = Namespace(
    manifest=MANIFEST, lenses=LENSES,
    max_train=32 if SMOKE_TEST else 1_000_000,
    max_eval=16 if SMOKE_TEST else 1_000_000,
    seed=42, max_steps=2 if SMOKE_TEST else -1,
    batch_size=2, gradient_accumulation=2 if SMOKE_TEST else 4,
    learning_rate=2e-4, warmup_steps=0,
    dcl_weight=0.1, infonce_weight=0.1, temperature=0.1,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
)
set_seed(args.seed)

## Load model and processor
This cell starts a fresh model. Do not rerun it between the two stages unless you intend to discard in-memory training progress.

In [4]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = build_model(MODEL_ID, full_finetune=False)
model.print_trainable_parameters()


[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.
Loading weights: 100%|███████████████████████████████████████████████████████| 471/471 [00:00<00:00, 22655.27it/s]


trainable params: 2,884,608 || all params: 259,369,536 || trainable%: 1.1122


## Stage 1: lens/organ alignment

In [5]:
stage1_trainer = run_stage(args, processor, model, "align", OUTPUT / "stage1", epochs=5)

Epoch,Training Loss,Validation Loss
0,24.054890,13.448557


## Stage 1 validation loss
This is loss-based validation on lens inputs, not original-image-only generative VQA accuracy.

In [6]:
stage1_metrics = stage1_trainer.evaluate()
stage1_metrics


Training Loss,Validation Loss,Epoch
24.054890,13.448557,0


{'eval_loss': 13.448556900024414}

## Stage 2: VQA alignment
Continues from the same in-memory model after stage 1.

In [7]:
stage2_trainer = run_stage(
    args, processor, model, "vqa", OUTPUT / "stage2",
    epochs=3,
)

Epoch,Training Loss,Validation Loss
0,11.641132,5.660913


In [8]:
stage2_metrics = stage2_trainer.evaluate()
stage2_metrics


Training Loss,Validation Loss,Epoch
11.641132,5.660913,0


{'eval_loss': 5.66091251373291}

## Inspect the training history

In [10]:
from IPython.display import display
display(stage2_trainer.state.log_history)


[{'loss': 13.571052551269531,
  'grad_norm': 58.28858184814453,
  'learning_rate': 0.0002,
  'epoch': 0.125,
  'step': 1},
 {'loss': 11.641132354736328,
  'grad_norm': 54.9015007019043,
  'learning_rate': 0.0001,
  'epoch': 0.25,
  'step': 2},
 {'eval_loss': 5.66091251373291,
  'eval_runtime': 25.5297,
  'eval_samples_per_second': 0.627,
  'eval_steps_per_second': 0.313,
  'epoch': 0.25,
  'step': 2},
 {'train_runtime': 52.4461,
  'train_samples_per_second': 0.153,
  'train_steps_per_second': 0.038,
  'total_flos': 37653918467328.0,
  'train_loss': 12.60609245300293,
  'epoch': 0.25,
  'step': 2},
 {'eval_loss': 5.66091251373291,
  'eval_runtime': 25.3491,
  'eval_samples_per_second': 0.631,
  'eval_steps_per_second': 0.316,
  'epoch': 0.25,
  'step': 2}]